# VitalRegistro — build Android no Colab

Execute em ordem. Requer repositório público, runtime Linux com Python 3.10–3.12 e espaço para SDK/NDK. Este notebook ainda não foi executado no Colab. Os erros interrompem a célula; não são ignorados. Não faça upload de dados pessoais.

In [ ]:
import sys, subprocess, pathlib, os
assert (3, 10) <= sys.version_info[:2] <= (3, 12), 'Runtime incompatível com este fluxo p4a master. Use Ubuntu 24.04/Python 3.12; veja docs/ANDROID.md.'
REPO_URL = 'https://github.com/SEU_USUARIO/vital-registro.git'
assert 'SEU_USUARIO' not in REPO_URL, 'Preencha REPO_URL com o repositório publicado.'
assert REPO_URL.startswith('https://github.com/') and '@' not in REPO_URL, 'Use URL pública sem credenciais.'
project = pathlib.Path('/content/vital-registro')
assert not project.exists(), 'Pasta já existe. Reutilize o clone nas próximas células ou reinicie o runtime.'
subprocess.run(['git', 'clone', REPO_URL, str(project)], check=True)
os.chdir(project)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
subprocess.run(['bash', 'scripts/setup_linux.sh'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## Licenças Android
Leia as [condições do Android SDK](https://developer.android.com/studio/terms). Só altere a opção abaixo para `True` se concordar. A alteração do spec fica apenas neste clone temporário.

In [ ]:
ACCEPT_ANDROID_SDK_LICENSES = False
assert ACCEPT_ANDROID_SDK_LICENSES, 'Leia as licenças antes de habilitar o aceite.'
spec = project / 'buildozer.spec'
content = spec.read_text(encoding='utf-8')
spec.write_text(content.replace('android.accept_sdk_license = False', 'android.accept_sdk_license = True'), encoding='utf-8')
env = os.environ.copy()
env['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
env['PATH'] = str(project / '.venv-build/bin') + ':' + env['JAVA_HOME'] + '/bin:' + env['PATH']
# Colab executa como root; confirma somente o aviso de root do Buildozer.
subprocess.run([str(project / '.venv-build/bin/buildozer'), '-v', 'android', 'debug'], env=env, input='y\n', text=True, check=True)


In [ ]:
from google.colab import files
apks = sorted((project / 'bin').glob('*.apk'))
assert apks, 'Nenhum APK gerado. Consulte a saída da compilação.'
for apk in apks:
    print(apk.name, apk.stat().st_size, 'bytes')
    files.download(str(apk))
# Registre este commit para fixar p4a.commit após validar o APK.
subprocess.run(['git', '-C', '.buildozer/android/platform/python-for-android', 'rev-parse', 'HEAD'], check=True)
